# Preparación de datos para la predicción de churn

## Objetivo

Este notebook documenta la preparación inicial del conjunto de datos Telco Customer Churn. Se revisan los tipos de datos, se convierte `TotalCharges` a formato numérico y se validan los cambios, conservando una copia de los datos originales.

## Alcance

El análisis exploratorio se presenta en `01_exploracion_inicial_churn.ipynb`. En esta etapa se carga el CSV, se trabaja sobre una copia y se documentan los valores faltantes identificados en `TotalCharges`. No se eliminan registros ni se imputan valores. La codificación de variables, el escalado, la partición de datos y el modelado se abordarán en etapas posteriores.

## 1. Carga de los datos

El archivo fuente se lee desde `data/raw`, dentro del proyecto. La ruta se construye de forma relativa al directorio del proyecto y se comprueba antes de cargar los datos. El conjunto contiene 7.043 registros y 21 columnas.

In [1]:
from pathlib import Path
import pandas as pd

ruta_csv = (
    Path.cwd()
    / "data"
    / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

print("Ruta encontrada:", ruta_csv.exists())

df_original = pd.read_csv(ruta_csv)

print("Dimensiones:", df_original.shape)
df_original.head()

Ruta encontrada: True
Dimensiones: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Copia de trabajo

Se crea `df_copia` para realizar la preparación sin modificar `df_original`. Ambos DataFrames conservan inicialmente las mismas dimensiones.

In [2]:
df_copia = df_original.copy()

print("Dimensiones del DataFrame original:", df_original.shape)
print("Dimensiones de la copia:", df_copia.shape)

Dimensiones del DataFrame original: (7043, 21)
Dimensiones de la copia: (7043, 21)


## 3. Revisión de `TotalCharges`

En el EDA se observó que `TotalCharges` está almacenada como texto y que hay 11 valores vacíos. Esos registros tienen `tenure` igual a 0 y `Churn` igual a `No`. Los vacíos no son detectados por `isna()` mientras sigan representados como texto.

La columna está almacenada como texto. Se identifican los valores vacíos y se inspeccionan sus registros asociados, sin eliminar filas ni reemplazar valores.

In [3]:
print("Tipo original de TotalCharges:", df_original["TotalCharges"].dtype)

mascara_vacios = df_original["TotalCharges"].str.strip().eq("")
print("Valores vacíos:", mascara_vacios.sum())

df_original.loc[
    mascara_vacios,
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

Tipo original de TotalCharges: str
Valores vacíos: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


## 4. Conversión numérica de `TotalCharges`

`TotalCharges` representa un cargo, por lo que su tipo numérico permitirá realizar cálculos más adelante.

Se conserva la columna de texto en `df_original` y se convierte una serie limpia en la copia de trabajo. Con `errors="coerce"`, los textos vacíos se representan como `NaN`. La conversión no elimina filas ni imputa valores; los 11 casos quedan pendientes de una decisión justificada en una etapa posterior.

In [4]:
total_charges_texto = df_original["TotalCharges"].copy()
total_charges_limpio = total_charges_texto.str.strip()
total_charges_numerico = pd.to_numeric(total_charges_limpio, errors="coerce")

vacios_texto = total_charges_limpio.eq("").sum()
faltantes_numericos = total_charges_numerico.isna().sum()
df_copia["TotalCharges"] = total_charges_numerico

print("Tipo original:", total_charges_texto.dtype)
print("Tipo después de la conversión:", df_copia["TotalCharges"].dtype)
print("Valores vacíos en el texto original:", vacios_texto)
print("Valores faltantes después de convertir:", faltantes_numericos)

Tipo original: str
Tipo después de la conversión: float64
Valores vacíos en el texto original: 11
Valores faltantes después de convertir: 11


## 5. Validación de la preparación

La validación confirma que `TotalCharges` es numérica, que sus 11 valores vacíos se representan como faltantes y que se conservaron las 7.043 filas y las 21 columnas.

In [5]:
resumen_validacion = pd.DataFrame({
    "Tipo de dato": df_copia.dtypes.astype(str),
    "Valores faltantes": df_copia.isna().sum()
})

resumen_validacion

,Tipo de dato,Valores faltantes
customerID,str,0
gender,str,0
SeniorCitizen,int64,0
Partner,str,0
Dependents,str,0
tenure,int64,0
PhoneService,str,0
MultipleLines,str,0
InternetService,str,0
OnlineSecurity,str,0


In [6]:
df_copia.shape

(7043, 21)

## 6. Función de las columnas

`customerID` se conserva en el conjunto preparado para identificar y rastrear registros, pero se separa de las características candidatas porque es un identificador. `Churn` es la variable objetivo. Las demás columnas son características candidatas; su inclusión final se definirá en la etapa de modelado.

In [7]:
columna_id = "customerID"
columna_objetivo = "Churn"

caracteristicas_candidatas = [
    columna
    for columna in df_copia.columns
    if columna not in [columna_id, columna_objetivo]
]

print("Identificador:", columna_id)
print("Objetivo:", columna_objetivo)
print("Características candidatas:", caracteristicas_candidatas)

Identificador: customerID
Objetivo: Churn
Características candidatas: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


## 7. Resumen del conjunto preparado

El resumen presenta las dimensiones, los tipos de datos y los valores faltantes de cada columna tras la conversión.

In [8]:
print("Dimensiones:", df_copia.shape)

resumen_final = pd.DataFrame({
    "Tipo de dato": df_copia.dtypes.astype(str),
    "Valores faltantes": df_copia.isna().sum()
})

resumen_final

Dimensiones: (7043, 21)


,Tipo de dato,Valores faltantes
customerID,str,0
gender,str,0
SeniorCitizen,int64,0
Partner,str,0
Dependents,str,0
tenure,int64,0
PhoneService,str,0
MultipleLines,str,0
InternetService,str,0
OnlineSecurity,str,0


## 8. Guardar el conjunto preparado

Para que la siguiente etapa pueda reutilizar esta preparación sin repetir el EDA ni la conversión de `TotalCharges`, se guarda la copia preparada como un artefacto CSV. El archivo fuente permanece intacto y los 11 valores faltantes se conservan como tales.

Ejecuta esta celda al terminar el notebook. El archivo generado será leído por `03_modelo_baseline_churn.ipynb`.

In [9]:
ruta_procesados = Path.cwd() / "data" / "processed"
ruta_procesados.mkdir(parents=True, exist_ok=True)

ruta_datos_preparados = ruta_procesados / "telco_churn_preparado.csv"
df_copia.to_csv(ruta_datos_preparados, index=False)

print("Archivo preparado guardado en:", ruta_datos_preparados)

Archivo preparado guardado en: c:\Users\user\Desktop\ESPECIALIZACION_SEMESTRE_2\PROYECTO\data\processed\telco_churn_preparado.csv


## 9. Conclusiones y decisiones pendientes

`TotalCharges` estaba almacenada como texto y ahora tiene tipo numérico (`float64`). Los 11 valores vacíos se convirtieron en valores faltantes (`NaN`). Se conservaron las 7.043 filas, verificado al comparar las dimensiones del DataFrame original y la copia preparada.

Los 11 registros con `tenure` igual a 0 y `TotalCharges` vacío se conservaron; no se eliminaron ni imputaron. En una etapa posterior se deberá decidir y justificar su tratamiento antes de usarlos en un modelo.